# Employee Attrition Prediction - EDA & Modeling
This notebook fulfills the requirements outlined in the Capstone Project.
We will perform EDA, feature engineering, and train Logistic Regression, Random Forest, and XGBoost models.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load dataset
df = pd.read_csv('../Dataset/dataset.csv')
df.head()

## 1. Data Preprocessing
Dropping irrelevant columns: `EmployeeCount`, `Over18`, `StandardHours`, `EmployeeNumber`

In [ ]:
df = df.drop(['EmployeeCount', 'Over18', 'StandardHours', 'EmployeeNumber'], axis=1, errors='ignore')
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})
df.info()

## 2. Exploratory Data Analysis (EDA)
Let's look at the correlation heatmap and cross-tabs of attrition vs key indicators (`OverTime`, `JobSatisfaction`, `MonthlyIncome`, `YearsAtCompany`).

In [ ]:
plt.figure(figsize=(10, 8))
# Select only numeric columns for correlation
numeric_df = df.select_dtypes(include=['int64', 'float64'])
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

# Cross-tabs
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

sns.countplot(x='OverTime', hue='Attrition', data=df, ax=axes[0, 0])
axes[0, 0].set_title('Attrition by OverTime')

sns.countplot(x='JobSatisfaction', hue='Attrition', data=df, ax=axes[0, 1])
axes[0, 1].set_title('Attrition by JobSatisfaction')

sns.boxplot(x='Attrition', y='MonthlyIncome', data=df, ax=axes[1, 0])
axes[1, 0].set_title('Attrition by Monthly Income')

sns.boxplot(x='Attrition', y='YearsAtCompany', data=df, ax=axes[1, 1])
axes[1, 1].set_title('Attrition by Years at Company')

plt.tight_layout()
plt.show()

## 3. Feature Engineering & Selection
Encoding categorical variables and preparing the dataset.

In [ ]:
categorical_cols = df.select_dtypes(include=['object', 'str']).columns.tolist()
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

X = df_encoded.drop('Attrition', axis=1)
y = df_encoded['Attrition']

from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

feat_importances = pd.Series(rf.feature_importances_, index=X.columns)
feat_importances.nlargest(10).plot(kind='barh')
plt.title('Top 10 Feature Importances')
plt.show()

## 4. Model Training & Evaluation
We'll train Logistic Regression, Random Forest, and XGBoost, using SMOTE to handle class imbalance. We will evaluate using Precision, Recall, F1 Score, and ROC-AUC.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score, f1_score
from imblearn.over_sampling import SMOTE

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

models = {
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(random_state=42, n_estimators=100),
    "XGBoost": XGBClassifier(random_state=42, eval_metric='logloss')
}

for name, model in models.items():
    print(f"--- {name} ---")
    model.fit(X_train_resampled, y_train_resampled)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    print(classification_report(y_test, y_pred))
    print(f"ROC AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n")